# 03. ARIMA / SARIMAX 모델

## 모델 선택 이유
- 3주 단기 예측에서 **자기상관(관성)**이 가장 강한 신호
- SARIMAX는 금리, KOSPI 등 **외생변수(exogenous)**를 추가로 반영
- 통계적 신뢰구간(95% CI) 제공 → 과제 보고서에 활용

## 전략
1. ADF 검정으로 차분 차수(d) 결정
2. ACF/PACF로 p, q 범위 탐색
3. auto_arima(pmdarima)로 자동 최적 파라미터 탐색
4. Walk-forward validation으로 성능 평가
5. 5월 25일 예측값 + 신뢰구간 출력

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# pmdarima 없으면 설치
try:
    from pmdarima import auto_arima
    HAS_PMDARIMA = True
except ImportError:
    print('pmdarima 미설치 → pip install pmdarima')
    HAS_PMDARIMA = False

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')

TARGET_DISTRICTS = ['노원구', '은평구', '서대문구', '서초구', '강남구', '송파구']
PREDICT_DATE = '2026-05-25'

combined = pd.read_csv(DATA_RAW / 'combined_weekly.csv', index_col='date', parse_dates=True)
fin_cols  = [c for c in ['base_rate', 'mortgage_rate', 'kospi'] if c in combined.columns]
print(f'데이터 로드: {combined.shape}, 금융 변수: {fin_cols}')

## 1. ACF / PACF 분석

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sample_district = '강남구' if '강남구' in combined.columns else TARGET_DISTRICTS[0]
series = combined[sample_district].dropna()
diff1  = series.diff().dropna()

plot_acf(series,  lags=40, ax=axes[0, 0], title=f'{sample_district} ACF (원시)')
plot_pacf(series, lags=40, ax=axes[0, 1], title=f'{sample_district} PACF (원시)')
plot_acf(diff1,   lags=40, ax=axes[1, 0], title=f'{sample_district} ACF (1차차분)')
plot_pacf(diff1,  lags=40, ax=axes[1, 1], title=f'{sample_district} PACF (1차차분)')

plt.tight_layout()
plt.show()

## 2. auto_arima로 최적 파라미터 탐색

In [ ]:
def find_best_order(series, exog=None, seasonal=True, m=52):
    """pmdarima auto_arima로 최적 (p,d,q)(P,D,Q) 탐색."""
    if not HAS_PMDARIMA:
        print('pmdarima 없음 → 기본값 (1,1,1)(0,0,0) 사용')
        return (1, 1, 1), (0, 0, 0, 0)

    model = auto_arima(
        series, exogenous=exog,
        start_p=0, max_p=4, start_q=0, max_q=4,
        d=None, max_d=2,
        seasonal=seasonal, m=m,
        stepwise=True, information_criterion='aic',
        error_action='ignore', suppress_warnings=True
    )
    print(f'  Best order: {model.order}, seasonal: {model.seasonal_order}, AIC: {model.aic():.2f}')
    return model.order, model.seasonal_order


best_orders = {}
for district in TARGET_DISTRICTS:
    if district not in combined.columns:
        continue
    print(f'\n[{district}] 최적 파라미터 탐색...')
    series = combined[district].dropna()
    exog   = combined[fin_cols].dropna().reindex(series.index).ffill().bfill() if fin_cols else None
    order, seasonal_order = find_best_order(series, exog)
    best_orders[district] = {'order': order, 'seasonal_order': seasonal_order}

## 3. Walk-forward Validation

In [ ]:
def walk_forward_eval(series, exog, order=(1,1,1), seasonal_order=(0,0,0,0),
                      test_weeks=12, predict_steps=1):
    """마지막 test_weeks 주를 한 주씩 롤링 예측하여 MAE/RMSE 계산."""
    n = len(series)
    split = n - test_weeks

    actuals, predictions = [], []
    for i in range(test_weeks):
        train_y = series.iloc[:split + i]
        train_x = exog.iloc[:split + i] if exog is not None else None
        test_x  = exog.iloc[split + i: split + i + predict_steps] if exog is not None else None

        try:
            model = SARIMAX(train_y, exog=train_x, order=order,
                            seasonal_order=seasonal_order,
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(disp=False)
            pred = model.forecast(steps=predict_steps, exog=test_x)
            predictions.append(float(pred.iloc[0]))
            actuals.append(float(series.iloc[split + i]))
        except Exception as e:
            print(f'  step {i} 실패: {e}')

    mae  = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mape = np.mean(np.abs((np.array(actuals) - np.array(predictions)) / np.array(actuals))) * 100
    return {'mae': mae, 'rmse': rmse, 'mape': mape,
            'actuals': actuals, 'predictions': predictions}


sarimax_scores = {}
for district in TARGET_DISTRICTS:
    if district not in combined.columns:
        continue
    series = combined[district].dropna()
    exog   = combined[fin_cols].reindex(series.index).ffill().bfill() if fin_cols else None
    params = best_orders.get(district, {'order': (1,1,1), 'seasonal_order': (0,0,0,0)})

    print(f'[{district}] Walk-forward 평가 중...')
    scores = walk_forward_eval(series, exog, **params)
    sarimax_scores[district] = scores
    print(f'  MAE={scores["mae"]:.4f}, RMSE={scores["rmse"]:.4f}, MAPE={scores["mape"]:.2f}%')

pd.DataFrame({k: {m: v for m, v in v.items() if m in ['mae','rmse','mape']}
              for k, v in sarimax_scores.items()}).T

## 4. 5월 25일 예측 (SARIMAX)

In [ ]:
def predict_future_sarimax(series, exog, order, seasonal_order, steps=3):
    """전체 데이터로 모델 학습 후 steps주 앞 예측."""
    model = SARIMAX(series, exog=exog, order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False).fit(disp=False)

    # 미래 외생변수: 마지막 값을 그대로 유지 (단기 예측이므로 합리적)
    if exog is not None:
        future_exog = pd.DataFrame(
            [exog.iloc[-1].values] * steps,
            columns=exog.columns
        )
    else:
        future_exog = None

    forecast = model.get_forecast(steps=steps, exog=future_exog)
    pred_mean = forecast.predicted_mean
    conf_int  = forecast.conf_int(alpha=0.05)  # 95% CI
    return pred_mean, conf_int


sarimax_predictions = {}
future_dates = pd.date_range(
    start=combined.index[-1] + pd.Timedelta(weeks=1),
    periods=3, freq='W-MON'
)

for district in TARGET_DISTRICTS:
    if district not in combined.columns:
        continue
    series = combined[district].dropna()
    exog   = combined[fin_cols].reindex(series.index).ffill().bfill() if fin_cols else None
    params = best_orders.get(district, {'order': (1,1,1), 'seasonal_order': (0,0,0,0)})

    pred, ci = predict_future_sarimax(series, exog, steps=3, **params)
    pred.index = future_dates
    ci.index   = future_dates
    sarimax_predictions[district] = {'pred': pred, 'ci': ci}

# 5월 25일 예측값 출력
target_date = pd.Timestamp(PREDICT_DATE)
closest_date = min(future_dates, key=lambda d: abs((d - target_date).days))

print(f'\n=== SARIMAX 예측 결과 ({closest_date.date()}) ===')
results = []
for district, data in sarimax_predictions.items():
    pred_val = data['pred'].loc[closest_date]
    ci_low   = data['ci'].iloc[list(future_dates).index(closest_date), 0]
    ci_high  = data['ci'].iloc[list(future_dates).index(closest_date), 1]
    results.append({'구': district, 'SARIMAX 예측': round(pred_val, 2),
                    '95% CI 하한': round(ci_low, 2), '95% CI 상한': round(ci_high, 2)})

result_df = pd.DataFrame(results)
result_df.to_csv(DATA_PROC / 'sarimax_predictions.csv', index=False)
print(result_df.to_string(index=False))

In [ ]:
# 예측 시각화
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
colors = plt.cm.tab10.colors

for ax, district, color in zip(axes.flat, TARGET_DISTRICTS, colors):
    if district not in combined.columns:
        continue
    series = combined[district].dropna()
    data   = sarimax_predictions[district]

    ax.plot(series.index[-52:], series.iloc[-52:], color=color, label='실제값')
    ax.plot(data['pred'].index, data['pred'].values,
            'r--o', markersize=6, label='SARIMAX 예측')
    ax.fill_between(data['ci'].index,
                    data['ci'].iloc[:, 0], data['ci'].iloc[:, 1],
                    alpha=0.2, color='red', label='95% CI')
    ax.axvline(pd.Timestamp(PREDICT_DATE), color='gray', linestyle=':', alpha=0.7)
    ax.set_title(district)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f'SARIMAX 예측 ({PREDICT_DATE})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_PROC / 'sarimax_forecast.png', dpi=150, bbox_inches='tight')
plt.show()